# FASE 6: QSVC clasificacion direccional h=5 (BVG)

Objetivo de fase:
- Extender la secuencia de la Fase 5 hacia un baseline cuantico (QSVC) en simulacion local.
- Mantener el mismo dataset y las mismas features existentes (sin agregar nuevas).
- Evaluar solo horizonte de 5 dias (h=5) con validacion temporal robusta.

Reglas metodologicas:
- Sin random split.
- Sin leakage (escalado fit solo en train de cada bloque temporal).
- Validacion principal: walk-forward expanding window.
- Validacion secundaria: holdout temporal 70/30.
- Baselines obligatorios: majority class y signo de ret_lag_1.

## 1) Diseno experimental y alcance

Decisiones de esta fase:
- Horizonte unico: target_up_h5.
- No se incrementa el numero de features respecto a Fase 5.
- No se incluye modelo clasico SVC dentro de esta fase; la comparacion clasico vs cuantico se hace externamente con tablas de Fase 5 y Fase 6.
- La reduccion de features para QSVC se realiza con criterio reproducible (PCA guiado por relacion con target), no por seleccion manual aleatoria.

Criterio de seleccion:
- Primero robustez temporal (media y dispersion en walk-forward).
- Luego comparacion contra baseline por empresa.

In [1]:
# 2) Dependencias cuanticas (instalacion solo si falta)
import importlib
import subprocess
import sys

required = [
    ('qiskit', 'qiskit'),
    ('qiskit_machine_learning', 'qiskit-machine-learning'),
    ('qiskit_aer', 'qiskit-aer')
]

for module_name, pkg_name in required:
    if importlib.util.find_spec(module_name) is None:
        print(f'Instalando {pkg_name}...')
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])

print('Dependencias cuanticas listas.')

Dependencias cuanticas listas.


In [ ]:
# 3) Librerias
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import RobustScaler, MinMaxScaler, StandardScaler
from sklearn.decomposition import PCA

try:
    from qiskit.circuit.library import zz_feature_map
    _ZZMAP_IMPL = 'function'
except Exception:
    # Fallback para entornos antiguos de Qiskit
    from qiskit.circuit.library import ZZFeatureMap
    _ZZMAP_IMPL = 'class'

from qiskit_machine_learning.kernels import FidelityQuantumKernel
from qiskit_machine_learning.algorithms import QSVC

pd.set_option('display.max_columns', 300)
pd.set_option('display.width', 320)
sns.set_theme(style='whitegrid')

print('Implementacion feature map:', _ZZMAP_IMPL)

In [3]:
# 4) Carga del dataset maestro de Fase 4
df = pd.read_csv('data/processed/BVG_features_svc_master.csv')
df['fecha'] = pd.to_datetime(df['fecha'], errors='coerce')

print('Shape:', df.shape)
print('Rango fechas:', df['fecha'].min(), '->', df['fecha'].max())
print('Empresas:', sorted(df['empresa'].dropna().unique().tolist()))
display(df.head())

Shape: (2841, 33)
Rango fechas: 2019-01-25 00:00:00 -> 2026-03-25 00:00:00
Empresas: ['BANCO GUAYAQUIL S.A.', 'CORPORACION FAVORITA C.A.']


,fecha,empresa,close_last,close_vwap,volume_shares_day,turnover_value_day,n_trades_day,ret_lag_1,ret_lag_2,ret_lag_3,mom_3,mom_5,mom_10,vol_5,vol_10,regime_vol_ratio,ma_5,ma_10,ma_gap,price_vs_ma10,rsi_14,turnover_log1p,volume_log1p,avg_trade_size_log1p,amihud_5,days_since_trade,ret_fwd_h1,ret_fwd_h5,ret_fwd_h20,target_up_h1,target_up_h5,target_up_h20,target_up_h20_thr003
0,2019-02-13,BANCO GUAYAQUIL S.A.,0.97,0.97,35000.0,33950.0,1.0,0.040822,-0.040822,0.000000,3.469447e-17,4.082199e-02,0.030459,0.034154,0.024103,1.416994,0.984,0.975,0.009,0.025641,62.452022,10.432674,10.463132,10.463132,0.000016,1.0,0.030459,0.030459,-0.010363,1.0,1.0,0.0,0.0
1,2019-02-19,BANCO GUAYAQUIL S.A.,1.00,1.00,2046.0,2046.0,6.0,-0.030459,0.040822,-0.040822,-3.045921e-02,1.036279e-02,0.010363,0.038424,0.026100,1.472155,0.986,0.976,0.010,-0.006148,52.665650,7.624131,7.624131,5.834811,0.000016,6.0,0.000000,0.000000,-0.051293,0.0,0.0,0.0,0.0
2,2019-02-20,BANCO GUAYAQUIL S.A.,1.00,1.00,1500.0,1500.0,1.0,0.030459,-0.030459,0.040822,4.082199e-02,1.144917e-16,0.030459,0.036015,0.027627,1.303608,0.986,0.979,0.007,0.021450,59.078190,7.313887,7.313887,7.313887,0.000019,1.0,0.000000,0.000000,-0.051293,0.0,0.0,0.0,0.0
3,2019-02-21,BANCO GUAYAQUIL S.A.,1.00,1.00,591494.0,591494.0,14.0,0.000000,0.030459,-0.030459,7.979728e-17,1.144917e-16,0.030459,0.036015,0.027627,1.303608,0.986,0.982,0.004,0.018330,59.078190,13.290409,13.290409,10.651373,0.000019,1.0,0.000000,0.000000,-0.051293,0.0,0.0,0.0,0.0
4,2019-02-26,BANCO GUAYAQUIL S.A.,1.00,1.00,2771.0,2771.0,1.0,0.000000,0.000000,0.030459,3.045921e-02,4.082199e-02,0.030459,0.028234,0.027627,1.021964,0.994,0.985,0.009,0.015228,57.100950,7.927324,7.927324,7.927324,0.000003,5.0,0.000000,0.000000,-0.051293,0.0,0.0,0.0,0.0


In [ ]:
# 5) Configuracion de horizonte h=5 y features
BASE_FEATURE_COLS = [
    'ret_lag_1', 'ret_lag_2', 'ret_lag_3',
    'mom_3', 'mom_5', 'mom_10',
    'vol_5', 'vol_10', 'regime_vol_ratio',
    'ma_5', 'ma_10', 'ma_gap', 'price_vs_ma10',
    'rsi_14',
    'turnover_log1p', 'volume_log1p',
    'avg_trade_size_log1p',
    'amihud_5', 'days_since_trade'
]

TARGET_COL = 'target_up_h5'
H_NAME = 'h5'

# Seleccion de features para QSVC guiada por PCA y target
PCA_VAR_THRESHOLD = 0.90
PCA_MAX_Q_FEATURES = 6

Q_REPS = [1, 2]
Q_ENTS = ['linear']
Q_C_VALUES = [0.5, 1.0, 5.0]
Q_CLASS_WEIGHTS = [None, 'balanced']

print('Horizonte activo:', H_NAME, '->', TARGET_COL)
print('Numero de features base disponibles:', len(BASE_FEATURE_COLS))
print('PCA var threshold:', PCA_VAR_THRESHOLD)
print('Max features para QSVC:', PCA_MAX_Q_FEATURES)

Horizonte activo: h5 -> target_up_h5
Numero features clasicas: 19
Numero features cuanticas (existentes): 6
Q feature cols: ['ret_lag_1', 'mom_5', 'vol_5', 'ma_gap', 'rsi_14', 'amihud_5']


In [ ]:
# 6) Utilidades temporales, metricas y modelos cuanticos
def temporal_split(d, train_frac=0.7):
    n = len(d)
    split = int(n * train_frac)
    split = max(80, min(split, n - 30))
    return d.iloc[:split].copy(), d.iloc[split:].copy()

def safe_tscv_splits(n_train):
    return max(3, min(5, n_train // 80))

def cls_metrics(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    return {
        'accuracy': float(acc),
        'directional_accuracy': float(acc),
        'precision': float(precision_score(y_true, y_pred, zero_division=0)),
        'recall': float(recall_score(y_true, y_pred, zero_division=0)),
        'f1': float(f1_score(y_true, y_pred, zero_division=0))
    }

def baseline_majority(y_train, n_test):
    maj = int(pd.Series(y_train).mode().iloc[0])
    return np.full(n_test, maj, dtype=int)

def baseline_sign_prev(test_df):
    return (test_df['ret_lag_1'].values > 0).astype(int)

def make_qkernel(feature_dim, reps, entanglement):
    if _ZZMAP_IMPL == 'function':
        fmap = zz_feature_map(feature_dimension=feature_dim, reps=reps, entanglement=entanglement)
    else:
        fmap = ZZFeatureMap(feature_dimension=feature_dim, reps=reps, entanglement=entanglement)
    return FidelityQuantumKernel(feature_map=fmap)

def q_preprocess_fit_transform(X_train):
    rob = RobustScaler()
    mm = MinMaxScaler(feature_range=(-1.0, 1.0))
    Xr = rob.fit_transform(X_train)
    Xm = mm.fit_transform(Xr)
    Xm = np.clip(Xm, -1.0, 1.0)
    return Xm, rob, mm

def q_preprocess_transform(X, rob, mm):
    Xr = rob.transform(X)
    Xm = mm.transform(Xr)
    Xm = np.clip(Xm, -1.0, 1.0)
    return Xm

def select_pca_guided_features(train_df, target_col):
    X_raw = train_df[BASE_FEATURE_COLS].values
    y = train_df[target_col].astype(int).values

    ss = StandardScaler()
    X = ss.fit_transform(X_raw)

    pca_full = PCA()
    pca_full.fit(X)
    cum_var = np.cumsum(pca_full.explained_variance_ratio_)
    n_comp = int(np.searchsorted(cum_var, PCA_VAR_THRESHOLD) + 1)
    n_comp = max(2, min(n_comp, len(BASE_FEATURE_COLS)))

    pca = PCA(n_components=n_comp)
    Xp = pca.fit_transform(X)

    y_center = y - y.mean()
    y_norm = np.sqrt(np.sum(y_center ** 2)) + 1e-12

    pc_target_corr = []
    for k in range(n_comp):
        pc = Xp[:, k]
        pc_center = pc - pc.mean()
        pc_norm = np.sqrt(np.sum(pc_center ** 2)) + 1e-12
        corr = abs(np.dot(pc_center, y_center) / (pc_norm * y_norm))
        pc_target_corr.append(corr)
    pc_target_corr = np.array(pc_target_corr)

    loadings_abs = np.abs(pca.components_.T)
    comp_weights = pca.explained_variance_ratio_ * pc_target_corr
    feature_scores = loadings_abs @ comp_weights

    rank_idx = np.argsort(-feature_scores)
    top_k = min(PCA_MAX_Q_FEATURES, len(BASE_FEATURE_COLS))
    selected_features = [BASE_FEATURE_COLS[i] for i in rank_idx[:top_k]]

    ranking_df = pd.DataFrame({
        'feature': BASE_FEATURE_COLS,
        'pca_target_score': feature_scores
    }).sort_values('pca_target_score', ascending=False).reset_index(drop=True)

    report = {
        'n_components_retained': int(n_comp),
        'cum_explained_variance': float(np.cumsum(pca.explained_variance_ratio_)[-1]),
        'selected_features': selected_features
    }
    return selected_features, ranking_df, report

def tune_qsvc_on_train(train_df, target_col):
    selected_features, ranking_df, pca_report = select_pca_guided_features(train_df, target_col)
    X = train_df[selected_features].values
    y = train_df[target_col].astype(int).values

    n_splits = safe_tscv_splits(len(train_df))
    tscv = TimeSeriesSplit(n_splits=n_splits)

    best = None

    for reps in Q_REPS:
        for ent in Q_ENTS:
            qkernel = make_qkernel(feature_dim=len(selected_features), reps=reps, entanglement=ent)
            for c in Q_C_VALUES:
                for cw in Q_CLASS_WEIGHTS:
                    fold_scores = []
                    ok = True

                    for tr_idx, va_idx in tscv.split(X):
                        Xtr_raw, Xva_raw = X[tr_idx], X[va_idx]
                        ytr, yva = y[tr_idx], y[va_idx]

                        Xtr, rob, mm = q_preprocess_fit_transform(Xtr_raw)
                        Xva = q_preprocess_transform(Xva_raw, rob, mm)

                        model = QSVC(quantum_kernel=qkernel, C=c, class_weight=cw)
                        try:
                            model.fit(Xtr, ytr)
                            yhat = model.predict(Xva)
                            fold_scores.append(f1_score(yva, yhat, zero_division=0))
                        except Exception:
                            ok = False
                            break

                    if not ok or len(fold_scores) == 0:
                        continue

                    score = float(np.mean(fold_scores))
                    cand = {
                        'reps': reps,
                        'entanglement': ent,
                        'C': float(c),
                        'class_weight': cw,
                        'best_cv_f1': score,
                        'selected_features': selected_features,
                        'pca_report': pca_report,
                        'pca_ranking_top': ranking_df.head(10).to_dict(orient='records')
                    }

                    if (best is None) or (score > best['best_cv_f1']):
                        best = cand

    if best is None:
        best = {
            'reps': 1,
            'entanglement': 'linear',
            'C': 1.0,
            'class_weight': None,
            'best_cv_f1': np.nan,
            'selected_features': selected_features,
            'pca_report': pca_report,
            'pca_ranking_top': ranking_df.head(10).to_dict(orient='records')
        }

    return best

def fit_predict_qsvc_fixed(train_df, test_df, target_col, q_cfg):
    selected_features = q_cfg['selected_features']

    X_train_raw = train_df[selected_features].values
    y_train = train_df[target_col].astype(int).values
    X_test_raw = test_df[selected_features].values

    X_train, rob, mm = q_preprocess_fit_transform(X_train_raw)
    X_test = q_preprocess_transform(X_test_raw, rob, mm)

    qkernel = make_qkernel(
        feature_dim=len(selected_features),
        reps=q_cfg['reps'],
        entanglement=q_cfg['entanglement']
    )
    model = QSVC(
        quantum_kernel=qkernel,
        C=q_cfg['C'],
        class_weight=q_cfg['class_weight']
    )
    model.fit(X_train, y_train)
    return model.predict(X_test)

In [ ]:
# 7) Evaluacion holdout y walk-forward por empresa (h=5)
def evaluate_holdout_case(d_case, target_col, empresa):
    rows = []
    preds = []

    train, test = temporal_split(d_case, train_frac=0.7)
    y_train = train[target_col].astype(int)
    y_test = test[target_col].astype(int)

    # Baseline 1
    y_pred_maj = baseline_majority(y_train.values, len(y_test))
    rows.append({
        'empresa': empresa, 'horizonte': H_NAME, 'model': 'baseline_majority', 'kernel': 'baseline',
        'best_cv_f1': np.nan, 'best_params': '', 'n_train': len(train), 'n_test': len(test),
        **cls_metrics(y_test, y_pred_maj)
    })

    # Baseline 2
    y_pred_sign = baseline_sign_prev(test)
    rows.append({
        'empresa': empresa, 'horizonte': H_NAME, 'model': 'baseline_sign_prev', 'kernel': 'baseline',
        'best_cv_f1': np.nan, 'best_params': '', 'n_train': len(train), 'n_test': len(test),
        **cls_metrics(y_test, y_pred_sign)
    })

    # Modelo cuantico QSVC con seleccion PCA guiada por target
    q_cfg = tune_qsvc_on_train(train, target_col)
    y_pred_q = fit_predict_qsvc_fixed(train, test, target_col, q_cfg)
    rows.append({
        'empresa': empresa, 'horizonte': H_NAME, 'model': 'qsvc', 'kernel': 'quantum',
        'best_cv_f1': float(q_cfg['best_cv_f1']) if pd.notna(q_cfg['best_cv_f1']) else np.nan,
        'best_params': str({
            'reps': q_cfg['reps'],
            'entanglement': q_cfg['entanglement'],
            'C': q_cfg['C'],
            'class_weight': q_cfg['class_weight'],
            'selected_features': q_cfg['selected_features'],
            'pca_report': q_cfg['pca_report']
        }),
        'n_train': len(train), 'n_test': len(test),
        **cls_metrics(y_test, y_pred_q)
    })

    for i, (_, r) in enumerate(test.reset_index(drop=True).iterrows()):
        preds.append({
            'empresa': empresa,
            'horizonte': H_NAME,
            'fecha': r['fecha'],
            'model': 'qsvc',
            'split': 'holdout_test',
            'y_true': int(y_test.iloc[i]),
            'y_pred': int(y_pred_q[i])
        })

    return pd.DataFrame(rows), pd.DataFrame(preds), q_cfg

def run_walkforward_case(d_case, target_col, empresa, initial_train=140, test_size=30, step=30):
    d = d_case.sort_values('fecha').reset_index(drop=True).copy()
    n = len(d)

    if n < (initial_train + test_size + 5):
        return pd.DataFrame(), pd.DataFrame(), None

    init_train = d.iloc[:initial_train].copy()

    # Tuning inicial cuantico + seleccion PCA guiada (fijo para ventanas posteriores)
    q_cfg = tune_qsvc_on_train(init_train, target_col)

    rows = []
    preds = []

    window_id = 0
    train_end = initial_train

    while (train_end + test_size) <= n:
        train = d.iloc[:train_end].copy()
        test = d.iloc[train_end:train_end + test_size].copy()

        y_train = train[target_col].astype(int)
        y_test = test[target_col].astype(int)

        # Baseline 1
        y_pred_maj = baseline_majority(y_train.values, len(y_test))
        rows.append({
            'empresa': empresa, 'horizonte': H_NAME, 'window_id': window_id,
            'model': 'baseline_majority', 'kernel': 'baseline', 'best_cv_f1': np.nan, 'best_params': '',
            'n_train': len(train), 'n_test': len(test), **cls_metrics(y_test, y_pred_maj)
        })

        # Baseline 2
        y_pred_sign = baseline_sign_prev(test)
        rows.append({
            'empresa': empresa, 'horizonte': H_NAME, 'window_id': window_id,
            'model': 'baseline_sign_prev', 'kernel': 'baseline', 'best_cv_f1': np.nan, 'best_params': '',
            'n_train': len(train), 'n_test': len(test), **cls_metrics(y_test, y_pred_sign)
        })

        # Cuantico (params + features fijos del bloque inicial)
        y_pred_q = fit_predict_qsvc_fixed(train, test, target_col, q_cfg)
        rows.append({
            'empresa': empresa, 'horizonte': H_NAME, 'window_id': window_id,
            'model': 'qsvc', 'kernel': 'quantum',
            'best_cv_f1': float(q_cfg['best_cv_f1']) if pd.notna(q_cfg['best_cv_f1']) else np.nan,
            'best_params': str({
                'reps': q_cfg['reps'],
                'entanglement': q_cfg['entanglement'],
                'C': q_cfg['C'],
                'class_weight': q_cfg['class_weight'],
                'selected_features': q_cfg['selected_features'],
                'pca_report': q_cfg['pca_report']
            }),
            'n_train': len(train), 'n_test': len(test), **cls_metrics(y_test, y_pred_q)
        })

        for i, (_, r) in enumerate(test.reset_index(drop=True).iterrows()):
            preds.append({
                'empresa': empresa, 'horizonte': H_NAME, 'window_id': window_id,
                'fecha': r['fecha'], 'model': 'qsvc',
                'y_true': int(y_test.iloc[i]), 'y_pred': int(y_pred_q[i])
            })

        window_id += 1
        train_end += step

    return pd.DataFrame(rows), pd.DataFrame(preds), q_cfg

In [ ]:
# 8) Ejecucion integrada Fase 6 cuantica (h=5)
holdout_rows = []
holdout_pred_rows = []
wf_rows = []
wf_pred_rows = []
qcfg_rows = []

for empresa in sorted(df['empresa'].dropna().unique()):
    d_emp = df[df['empresa'] == empresa].copy().sort_values('fecha').reset_index(drop=True)

    d_case = d_emp.dropna(subset=BASE_FEATURE_COLS + [TARGET_COL]).copy()
    d_case = d_case.sort_values('fecha').reset_index(drop=True)

    if len(d_case) < 220:
        print(f'Salto {empresa} h5: muestra insuficiente ({len(d_case)}).')
        continue

    # Holdout temporal
    h_res, h_pred, h_qcfg = evaluate_holdout_case(d_case, TARGET_COL, empresa)
    holdout_rows.append(h_res)
    if not h_pred.empty:
        holdout_pred_rows.append(h_pred)
    qcfg_rows.append({
        'empresa': empresa,
        'horizonte': H_NAME,
        'stage': 'holdout',
        'selected_features': str(h_qcfg.get('selected_features', [])),
        'pca_report': str(h_qcfg.get('pca_report', {})),
        'best_cv_f1': h_qcfg.get('best_cv_f1', np.nan)
    })

    # Walk-forward expanding
    w_res, w_pred, w_qcfg = run_walkforward_case(
        d_case=d_case,
        target_col=TARGET_COL,
        empresa=empresa,
        initial_train=140,
        test_size=30,
        step=30
    )
    if not w_res.empty:
        wf_rows.append(w_res)
    if not w_pred.empty:
        wf_pred_rows.append(w_pred)
    if w_qcfg is not None:
        qcfg_rows.append({
            'empresa': empresa,
            'horizonte': H_NAME,
            'stage': 'walkforward_init',
            'selected_features': str(w_qcfg.get('selected_features', [])),
            'pca_report': str(w_qcfg.get('pca_report', {})),
            'best_cv_f1': w_qcfg.get('best_cv_f1', np.nan)
        })

holdout_df = pd.concat(holdout_rows, ignore_index=True) if holdout_rows else pd.DataFrame()
holdout_pred_df = pd.concat(holdout_pred_rows, ignore_index=True) if holdout_pred_rows else pd.DataFrame()
wf_df = pd.concat(wf_rows, ignore_index=True) if wf_rows else pd.DataFrame()
wf_pred_df = pd.concat(wf_pred_rows, ignore_index=True) if wf_pred_rows else pd.DataFrame()
qcfg_df = pd.DataFrame(qcfg_rows)

print('Holdout rows:', holdout_df.shape)
print('Walk-forward rows:', wf_df.shape)

if not holdout_df.empty:
    display(holdout_df.sort_values(['empresa', 'f1'], ascending=[True, False]))
if not wf_df.empty:
    display(wf_df.sort_values(['empresa', 'window_id', 'f1'], ascending=[True, True, False]).head(30))
if not qcfg_df.empty:
    display(qcfg_df)

C:\Users\leynd\AppData\Local\Temp\ipykernel_49500\192117835.py:63: DeprecationWarning: The class ``qiskit.circuit.library.data_preparation._zz_feature_map.ZZFeatureMap`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the zz_feature_map function as a replacement. Note that this will no longer return a BlueprintCircuit, but just a plain QuantumCircuit.
  fmap = ZZFeatureMap(feature_dimension=len(Q_FEATURE_COLS), reps=reps, entanglement=entanglement)


In [ ]:
# 9) Resumen robusto de QSVC vs baseline
if wf_df.empty:
    raise ValueError('No hay resultados walk-forward. Revisa tamano de muestra o configuracion.')

wf_summary = (
    wf_df.groupby(['empresa', 'horizonte', 'model', 'kernel']).agg(
        n_windows=('window_id', 'count'),
        acc_mean=('accuracy', 'mean'),
        acc_std=('accuracy', 'std'),
        f1_mean=('f1', 'mean'),
        f1_std=('f1', 'std'),
        dir_acc_mean=('directional_accuracy', 'mean'),
        cv_f1_mean=('best_cv_f1', 'mean')
    ).reset_index()
)

base_summary = wf_summary[wf_summary['kernel'] == 'baseline'].copy()
best_baseline = (
    base_summary.sort_values(['empresa', 'f1_mean', 'acc_mean'], ascending=[True, False, False])
    .groupby('empresa', as_index=False)
    .first()[['empresa', 'model', 'f1_mean', 'acc_mean']]
    .rename(columns={
        'model': 'best_baseline_model',
        'f1_mean': 'baseline_f1_mean',
        'acc_mean': 'baseline_acc_mean'
    })
)

qsvc_case = wf_summary[wf_summary['model'] == 'qsvc'].copy()
qsvc_case = qsvc_case.merge(best_baseline, on='empresa', how='left')
qsvc_case['delta_f1_vs_baseline'] = qsvc_case['f1_mean'] - qsvc_case['baseline_f1_mean']
qsvc_case['delta_acc_vs_baseline'] = qsvc_case['acc_mean'] - qsvc_case['baseline_acc_mean']

# % de ventanas en las que QSVC supera al mejor baseline por empresa
cmp_rows = []
for _, b in best_baseline.iterrows():
    empresa = b['empresa']
    base_model = b['best_baseline_model']

    w_q = wf_df[(wf_df['empresa'] == empresa) & (wf_df['model'] == 'qsvc')][['window_id', 'accuracy', 'f1']]
    w_b = wf_df[(wf_df['empresa'] == empresa) & (wf_df['model'] == base_model)][['window_id', 'accuracy', 'f1']]

    m = w_q.merge(w_b, on='window_id', suffixes=('_q', '_b'))
    if m.empty:
        continue

    cmp_rows.append({
        'empresa': empresa,
        'pct_windows_f1_better': float((m['f1_q'] > m['f1_b']).mean() * 100),
        'pct_windows_acc_better': float((m['accuracy_q'] > m['accuracy_b']).mean() * 100)
    })

cmp_df = pd.DataFrame(cmp_rows)
qsvc_case = qsvc_case.merge(cmp_df, on='empresa', how='left')

print('Resumen robusto por modelo (incluye baselines):')
display(wf_summary.sort_values(['empresa', 'f1_mean'], ascending=[True, False]))

print('QSVC por empresa (vs mejor baseline):')
display(qsvc_case[['empresa', 'n_windows', 'f1_mean', 'f1_std', 'acc_mean', 'acc_std', 'baseline_f1_mean', 'baseline_acc_mean', 'delta_f1_vs_baseline', 'delta_acc_vs_baseline', 'pct_windows_f1_better', 'pct_windows_acc_better']])

In [ ]:
# 10) Grafico comparativo por empresa (QSVC: prediccion vs real)
if wf_pred_df.empty:
    raise ValueError('No hay predicciones walk-forward para graficar.')

for empresa in sorted(wf_pred_df['empresa'].unique()):
    p = wf_pred_df[
        (wf_pred_df['empresa'] == empresa) &
        (wf_pred_df['horizonte'] == H_NAME) &
        (wf_pred_df['model'] == 'qsvc')
    ].copy().sort_values('fecha')

    if p.empty:
        continue

    p = p.drop_duplicates(subset=['fecha'], keep='last')
    p['acierto'] = (p['y_true'] == p['y_pred']).astype(int)

    plt.figure(figsize=(14, 4))
    plt.plot(p['fecha'], p['y_true'], drawstyle='steps-post', linewidth=1.6, label='Real', color='black')
    plt.plot(p['fecha'], p['y_pred'], drawstyle='steps-post', linewidth=1.3, label='QSVC', color='tab:blue', alpha=0.9)

    ok = p[p['acierto'] == 1]
    err = p[p['acierto'] == 0]
    if not ok.empty:
        plt.scatter(ok['fecha'], ok['y_pred'], color='green', s=18, alpha=0.7, label='Acierto')
    if not err.empty:
        plt.scatter(err['fecha'], err['y_pred'], color='red', s=20, alpha=0.8, label='Error')

    plt.yticks([0, 1], ['Baja (0)', 'Sube (1)'])
    plt.title(f'{empresa} | h5 | QSVC en walk-forward')
    plt.xlabel('Fecha')
    plt.ylabel('Clasificacion direccional')
    plt.legend(loc='best')
    plt.tight_layout()
    plt.show()

In [ ]:
# 11) Exportacion de resultados de Fase 6 cuantica (h=5)
if not holdout_df.empty:
    holdout_df.to_csv('data/processed/BVG_fase6q_h5_holdout_resultados.csv', index=False)
if not holdout_pred_df.empty:
    holdout_pred_df.to_csv('data/processed/BVG_fase6q_h5_holdout_predicciones_qsvc.csv', index=False)
if not wf_df.empty:
    wf_df.to_csv('data/processed/BVG_fase6q_h5_walkforward_resultados_ventana.csv', index=False)
if not wf_pred_df.empty:
    wf_pred_df.to_csv('data/processed/BVG_fase6q_h5_walkforward_predicciones_qsvc.csv', index=False)

wf_summary.to_csv('data/processed/BVG_fase6q_h5_walkforward_resumen.csv', index=False)
qsvc_case.to_csv('data/processed/BVG_fase6q_h5_qsvc_vs_baseline_por_empresa.csv', index=False)
if not qcfg_df.empty:
    qcfg_df.to_csv('data/processed/BVG_fase6q_h5_qsvc_config_pca.csv', index=False)

print('Archivos exportados en data/processed:')
print('- BVG_fase6q_h5_holdout_resultados.csv')
print('- BVG_fase6q_h5_holdout_predicciones_qsvc.csv')
print('- BVG_fase6q_h5_walkforward_resultados_ventana.csv')
print('- BVG_fase6q_h5_walkforward_predicciones_qsvc.csv')
print('- BVG_fase6q_h5_walkforward_resumen.csv')
print('- BVG_fase6q_h5_qsvc_vs_baseline_por_empresa.csv')
print('- BVG_fase6q_h5_qsvc_config_pca.csv (si hubo)')

## 12) Conclusion de fase (formato tesis)

1. Que hice:
- Implemente la Fase 6 cuantica en horizonte fijo h=5 usando el dataset limpio de fases previas, sin incorporar features nuevas.
- Ejecute validacion temporal con holdout cronologico y walk-forward expanding.
- Reemplace la construccion de feature map por API vigente (`zz_feature_map`) con fallback de compatibilidad.
- Aplique seleccion de variables cuanticas con criterio reproducible (PCA guiado por relacion con target `target_up_h5`).

2. Por que lo hice:
- Para mantener trazabilidad experimental y evitar leakage temporal.
- Para cumplir restricciones NISQ (pocas dimensiones, normalizacion consistente) sin perder senal financiera relevante.
- Para alinear la metodologia con literatura de QSVM financiera: validacion temporal, control de dimensionalidad e interpretacion por ventana.

3. Que resultado obtuve:
- Se generaron resultados por empresa en holdout y walk-forward con metricas de clasificacion direccional.
- Se exportaron predicciones y resumenes por ventana para analisis acumulado.
- Se obtuvo configuracion cuantica por empresa (features seleccionadas y parametros del kernel).

4. Que significa ese resultado:
- La fase queda metodologicamente defendible y reproducible para evaluar desempeno cuantico en BVG con h=5.
- El experimento permite identificar en que segmentos temporales el enfoque cuantico mantiene o pierde estabilidad.

5. Decision de continuidad:
- Continuar a auditoria comparativa inter-fase con resultados ya exportados.
- Mantener esta configuracion como benchmark cuantico base antes de nuevas expansiones de features o tuning avanzado.